# Finetuning with the Python SDK

> **⚠️ Experimental Feature**: The Finetuning Harness is experimental and may change in future releases.

This notebook demonstrates how to configure and run finetuning using the NeMo Agent Toolkit Python SDK.

## What You'll Learn

- How to configure curriculum learning for progressive training
- How to set up finetuning with trainers, trajectory builders, and trainer adapters **as objects**
- How to run finetuning programmatically
- How to save finetuning configurations for CLI usage

## Key Concept: Object-Based Configuration

The SDK **requires** passing actual component objects (not string names):

```python
# Objects are required - provides type checking, autocomplete, and better errors
finetuning = NatFinetuner(
    trainer=ARTTrainer(),                    # Not "openpipe_art_trainer" string
    trajectory_builder=ARTTrajectoryBuilder(),
    trainer_adapter=ARTTrainerAdapter(),
    reward_function=my_evaluator,
    ...
)

# Names are computed automatically:
print(finetuning.trainer_name)  # "openpipe_art_trainer"
```

This provides IDE autocomplete, type checking, and better error messages.

## Prerequisites

Before running finetuning, you need:

1. **Training Backend**: A running training backend (e.g., OpenPipe ART server with GPU)
2. **LLM Endpoint**: An inference endpoint with log probability support
3. **Training Dataset**: Data in JSON/JSONL format
4. **Custom Evaluator**: An evaluator for computing rewards


## Step 1: Setup and Imports

First, let's import the necessary modules.


In [ ]:
# Standard library imports
import nest_asyncio

# SDK imports
from nat.agent.react_agent.register import NatReActAgent
from nat.llm.openai_llm import OpenAILLM
from nat.utils.sdk.nat_finetuner import CurriculumLearning
from nat.utils.sdk.nat_workflow import NatWorkflow

# For notebook async support
nest_asyncio.apply()

# Note: For actual finetuning, you'll need:
# 1. Plugin implementations (e.g., nvidia-nat-openpipe-art)
# 2. A custom evaluator that returns rewards
# See the examples/finetuning/ directory for complete implementations.

print("✓ Imports complete")


## Step 2: Understanding Finetuning Configuration

The finetuning harness uses three main components:

| Component | Purpose |
|-----------|----------|
| **Trainer** | Orchestrates the finetuning loop across epochs |
| **Trajectory Builder** | Collects training data from workflow runs |
| **Trainer Adapter** | Submits trajectories to the training backend |

These components are registered via plugins (e.g., `nvidia-nat-openpipe-art` for OpenPipe ART).


## Step 3: Configure Curriculum Learning

Curriculum learning progressively introduces harder examples during training.


In [ ]:
# Configure curriculum learning for progressive training
# CurriculumLearning inherits from CurriculumLearningConfig - no field duplication!
curriculum = CurriculumLearning(
    enabled=True,
    initial_percentile=0.3,  # Start with easiest 30% of examples
    increment_percentile=0.2,  # Add 20% more examples each expansion
    expansion_interval=5,  # Expand curriculum every 5 epochs
    min_reward_diff=0.1,  # Skip groups with reward variance < 0.1
    sort_ascending=False,  # False = easy-to-hard curriculum
)

print("Curriculum Learning Configuration:")
print(f"  - Enabled: {curriculum.enabled}")
print(f"  - Start with: {curriculum.initial_percentile * 100:.0f}% of examples")
print(f"  - Add: {curriculum.increment_percentile * 100:.0f}% more every {curriculum.expansion_interval} epochs")


## Step 4: Configure Finetuning

Set up the main finetuning configuration.


In [ ]:
# Configure finetuning - REQUIRES passing objects, not string references
#
# NatFinetuner takes actual component objects which provide:
# - IDE autocomplete and type checking
# - Better error messages at construction time
# - Automatic name resolution for the underlying config

# Example with mock components (replace with actual plugin implementations)
#
# finetuning = NatFinetuner(
#     trainer=ARTTrainer(),                            # From nvidia-nat-openpipe-art
#     trajectory_builder=ARTTrajectoryBuilder(         # From nvidia-nat-openpipe-art
#         num_generations=2,
#     ),
#     trainer_adapter=ARTTrainerAdapter(               # From nvidia-nat-openpipe-art
#         backend=backend_config,
#     ),
#     reward_function=my_accuracy_evaluator,           # Your NatEvaluator subclass

#     # Training settings
#     num_epochs=10,
#     target_function_names=["<workflow>"],  # Extract from entire workflow
#     target_model_name=None,                 # Train all models (or specify one)
#     output_dir=Path(".tmp/nat/finetuning"),

#     # Curriculum learning
#     curriculum_learning=curriculum,
# )

# Note: To run this example, you need:
# 1. pip install nvidia-nat-openpipe-art (or another finetuning plugin)
# 2. A custom evaluator that computes rewards
# 3. A running training backend

print("To configure finetuning, pass objects directly:")
print("  - trainer: A NatTrainer subclass (e.g., ARTTrainer)")
print("  - trajectory_builder: A NatTrajectoryBuilder subclass")
print("  - trainer_adapter: A NatTrainerAdapter subclass")
print("  - reward_function: A NatEvaluator for computing rewards")
print()
print("After creating NatFinetuner, names are computed automatically:")
print("  - finetuning.trainer_name")
print("  - finetuning.trajectory_builder_name")
print("  - finetuning.trainer_adapter_name")
print("  - finetuning.reward_function_name")


## Step 5: Create a Complete Workflow

Now let's create a workflow with all the components needed for finetuning.


In [ ]:
import getpass
import os

from dotenv import load_dotenv

# Load environment variables from .env file if it exists
load_dotenv()

# Check for OpenAI API key
openai_api_key = os.environ.get("OPENAI_API_KEY")

if openai_api_key:
    print("✅ OPENAI_API_KEY loaded")
else:
    openai_api_key = getpass.getpass("Enter your OPENAI_API_KEY (or press Enter to skip): ")
    if openai_api_key:
        os.environ["OPENAI_API_KEY"] = openai_api_key
        print("✅ OPENAI_API_KEY set")
    else:
        print("⏭️ Skipping workflow creation (no API key)")

# Create LLM and workflow only if API key is available
workflow = None

if openai_api_key:
    from nat.utils.sdk.nat_env_var import NatEnvironmentVariable

    # Create LLM with log probabilities enabled
    # For finetuning, you need an endpoint that returns log probs
    llm = OpenAILLM(
        model_name="Qwen/Qwen2.5-7B-Instruct",
        base_url="http://localhost:8000/v1",  # Your vLLM endpoint
        api_key_env=NatEnvironmentVariable("OPENAI_API_KEY"),
    )

    # Create agent
    agent = NatReActAgent(
        llm=llm,
        tools=[],
        verbose=True,
    )

    # Create workflow
    workflow = NatWorkflow(entrypoint=agent)
    print("✓ Workflow created")
else:
    print("⚠️ Workflow not created - set OPENAI_API_KEY to continue")


## Step 6: Add Finetuning Configuration


In [ ]:
# Add finetuning configuration to workflow
# workflow.add_finetuning(finetuning)

# After adding finetuning, you can:
# - Save the workflow config: workflow.save_to_config_file("config.yaml")
# - Run finetuning: await workflow.finetune(dataset="data.json")

print("To add finetuning to a workflow:")
print("  workflow.add_finetuning(finetuning)")
print()
print("This enables:")
print("  - workflow.save_to_config_file() - includes finetuning config")
print("  - workflow.finetune() - runs finetuning programmatically")


## Step 7: Save Configuration for CLI

You can save the complete configuration to a YAML file and run finetuning via the CLI.

> **Note**: To run finetuning, you also need an evaluator configured. See the `09_evaluation.ipynb` notebook for details.


In [ ]:
# To save, you'd also need to add an evaluator:
# workflow.add_evaluator(evaluation)

# Then save configuration to YAML
# workflow.save_to_config_file("configs/finetuning_workflow.yaml")

print("To run finetuning via CLI:")
print("  nat finetune --config_file=configs/finetuning_workflow.yaml")


## Summary

In this notebook, you learned how to:

1. ✅ Configure curriculum learning for progressive training
2. ✅ Set up finetuning with the required components
3. ✅ Add finetuning configuration to a workflow
4. ✅ Save configuration for CLI usage

### Key Classes

| Class | Purpose |
|-------|----------|
| `NatFinetuner` | Main finetuning configuration |
| `CurriculumLearning` | Curriculum learning settings |
| `workflow.add_finetuning()` | Add finetuning to workflow |
| `workflow.finetune()` | Run finetuning programmatically |

### Next Steps

- See the `examples/finetuning/` directory for complete working examples
- See [Finetuning Concepts](../../../docs/source/improve-workflows/finetuning/concepts.md) for RL fundamentals
- See [OpenPipe ART Integration](../../../docs/source/improve-workflows/finetuning/rl_with_openpipe.md) for backend setup
